In [1]:
import pandas as pd

# загружаем данные
df = pd.read_csv('data/mental_health_digital_behavior_data.csv')

# словарь для перевода названий колонок
rename_map = {
    'daily_screen_time_min': 'Экранное время (мин)',
    'num_app_switches': 'Переключения приложений',
    'sleep_hours': 'Часы сна',
    'notification_count': 'Уведомления',
    'social_media_time_min': 'Соцсети (мин)',
    'mood_score': 'Настроение',
    'digital_wellbeing_score': 'Цифровое благополучие',  
    'anxiety_level': 'Уровень тревоги', 
    'focus_score': 'Концентрация'          
}

# считаем корреляции с тревогой
anxiety_corr = df.corr()['anxiety_level'].drop(['anxiety_level', 'focus_score', 'mood_score', 'digital_wellbeing_score'])

# считаем корреляции с концентрацией
focus_corr = df.corr()['focus_score'].drop(['focus_score', 'anxiety_level', 'mood_score', 'digital_wellbeing_score'])

# создаем таблицу для тревоги
anxiety_df = pd.DataFrame({
    'Фактор': [rename_map.get(f, f) for f in anxiety_corr.index],
    'Корреляция': anxiety_corr.values,
    'Показатель': 'Тревога'
})

# создаем таблицу для концентрации
focus_df = pd.DataFrame({
    'Фактор': [rename_map.get(f, f) for f in focus_corr.index],
    'Корреляция': focus_corr.values,
    'Показатель': 'Концентрация'
})

# объединеняем таблицы
result = pd.concat([anxiety_df, focus_df], ignore_index=True)

result.to_csv('data/correlations.csv', index=False)
print(result)

# ===== ЧАСТЬ 2: портрет топ-10% самых тревожных =====

# порог: значение тревоги, выше которого только 10% людей
threshold = df['anxiety_level'].quantile(0.9)

# самые тревожные людеи
top_anxious = df[df['anxiety_level'] >= threshold]

# средние показатели для этой группы
top_means = top_anxious[[
    'daily_screen_time_min',
    'num_app_switches',
    'sleep_hours',
    'notification_count',
    'social_media_time_min'
]].mean()

# средние показатели по всем людям для сравнения
all_means = df[[
    'daily_screen_time_min',
    'num_app_switches',
    'sleep_hours',
    'notification_count',
    'social_media_time_min'
]].mean()

# таблица сравнения
comparison = pd.DataFrame({
    'Показатель': [
        'Экранное время (мин)',
        'Переключения приложений',
        'Часы сна',
        'Уведомления',
        'Соцсети (мин)'
    ],
    'Топ 10% тревожных': top_means.values,
    'Средние по всем': all_means.values
})

# сохраненяем для Power BI
comparison.to_csv('data/top_anxious.csv', index=False)
print(comparison)

# ===== ЧАСТЬ 3: считаем разницу в % для карточек =====

# на сколько % топ 10% тревожных отличаются от средних
soc_diff = (top_means['social_media_time_min'] / all_means['social_media_time_min'] - 1) * 100
notif_diff = (top_means['notification_count'] / all_means['notification_count'] - 1) * 100

# Считаем средний digital wellbeing у топ-10% тревожных
top_wellbeing = top_anxious['digital_wellbeing_score'].mean()
# И у всех остальных
all_wellbeing = df['digital_wellbeing_score'].mean()
# Считаем разницу в %
wellbeing_diff = (top_wellbeing / all_wellbeing - 1) * 100

# Собираем таблицу для карточек — три строки
cards = pd.DataFrame({
    'Метрика': ['Соцсети', 'Уведомления', 'Самочувствие'],
    'Разница': [
        f'+{round(soc_diff, 1)}%',
        f'+{round(notif_diff, 1)}%',
        f'{round(wellbeing_diff, 1)}%'  # без + — скорее всего будет минус
    ]
})

cards.to_csv('data/cards.csv', index=False)
print(cards)

                    Фактор  Корреляция    Показатель
0     Экранное время (мин)    0.003953       Тревога
1  Переключения приложений   -0.028695       Тревога
2                 Часы сна    0.014952       Тревога
3              Уведомления    0.301004       Тревога
4            Соцсети (мин)    0.311640       Тревога
5     Экранное время (мин)   -0.306948  Концентрация
6  Переключения приложений   -0.279350  Концентрация
7                 Часы сна    0.010128  Концентрация
8              Уведомления   -0.336326  Концентрация
9            Соцсети (мин)   -0.053388  Концентрация
                Показатель  Топ 10% тревожных  Средние по всем
0     Экранное время (мин)         359.161603         360.4378
1  Переключения приложений          49.514768          49.8400
2                 Часы сна           6.522785           6.5574
3              Уведомления          87.666667          79.5120
4            Соцсети (мин)         135.644726         121.7718
        Метрика Разница
0       Соцсети